In [ ]:
library(ggplot2)  # For making plots
library(tidyr)    # For reshaping data (wide to long format)
library(dplyr)    # For data manipulation (select, filter, etc.)
library(e1071)    # For skewness() function


In [ ]:
# Define consistent color palette for wetland classes
class_colors <- c("Bog" = "#F8766D",      # Red/coral
                  "Fen" = "#7CAE00",      # Green
                  "Marsh" = "#00BFC4",    # Cyan/blue
                  "Swamp" = "#C77CFF")    # Purple

df <- read.csv("C:/Users/Leila/DropBox/MayoWetlands/wetland_alldata_2025_only_uscore.csv")


# Chemistry 

In [ ]:
# ==== GROUP 1: DISSOLVED METALS ====
metals <- c("Al", "Sb", "As", "Ba", "Be", "Bi", "B", "Cd", "Ca", "Cs", "Cr", 
            "Co", "Cu", "F", "Fe", "Pb", "Li", "Mg", "Mn", "Mo", "Ni", "P", 
            "K", "Rb", "Se", "Si", "Ag", "Na", "Sr", "S", "Te", "Tl", "Th", 
            "Sn", "Ti", "W", "U", "V", "Zn", "Zr")
metals <- metals[metals %in% names(df)]

metals_long <- df %>%
  select(Class, all_of(metals)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

ggplot(metals_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) + 
  facet_wrap(~ variable, scales = "free_y", ncol = 5) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Dissolved Metals by Wetland Class", y = "U-score")

ggsave("Figures/02_metals_distributions.png", width = 18, height = 12)

# ==== GROUP 2: NUTRIENTS ====
nutrients <- c("NH3_total", "TOC", "DOC", "P_total", "P_dissolved")
nutrients <- nutrients[nutrients %in% names(df)]

nutrients_long <- df %>%
  select(Class, all_of(nutrients)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

ggplot(nutrients_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) + 
  facet_wrap(~ variable, scales = "free_y", ncol = 3) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Nutrients by Wetland Class", y = "U-score")

ggsave("Figures/03_nutrients_distributions.png", width = 10, height = 6)

# ==== GROUP 3: GENERAL CHEMISTRY ====
general <- c("Alkalinity_total_CaCO3_mg.L", "TDS_mg.L", "Hardness", "SO4_mg.L")
general <- general[general %in% names(df)]

general_long <- df %>%
  select(Class, all_of(general)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

ggplot(general_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) + 
  facet_wrap(~ variable, scales = "free_y", ncol = 2) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "General Chemistry by Wetland Class", y = "U-score")

ggsave("Figures/04_general_chem_distributions.png", width = 8, height = 6)

# ==== GROUP 4: YSI PARAMETERS ====
ysi <- c("Dissolved_O_percent", "Dissolved_O_mgL", "pH", "SpC", "ORPmV")
ysi <- ysi[ysi %in% names(df)]

ysi_long <- df %>%
  select(Class, all_of(ysi)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

ggplot(ysi_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) + 
  facet_wrap(~ variable, scales = "free_y", ncol = 3) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "YSI Parameters by Wetland Class", y = "U-score")

ggsave("Figures/05_ysi_distributions.png", width = 10, height = 6)

cat("✓ Saved 4 chemistry group plots to Figures/\n")

#==========================================================================
# Calc Skewness 
#==========================================================================
# Calculate skewness for all chemistry variables

# Define all chemistry groups
all_chem <- c(metals, nutrients, general, ysi)

# Calculate skewness by class
skewness_results <- data.frame()

for(var in all_chem) {
  if(var %in% names(df)) {
    for(class in unique(df$Class)) {
      values <- df %>% filter(Class == class) %>% pull(var)
      values <- values[!is.na(values)]
      
      if(length(values) > 2) {  # Need at least 3 values
        skew_val <- skewness(values)
        
        skewness_results <- rbind(skewness_results, 
                                  data.frame(variable = var,
                                            class = class,
                                            skewness = skew_val,
                                            n = length(values)))
      }
    }
  }
}

# Flag highly skewed (|skewness| > 1 = moderate, >2 = severe)
skewness_results$severity <- ifelse(abs(skewness_results$skewness) > 2, "Severe",
                                     ifelse(abs(skewness_results$skewness) > 1, "Moderate", "Mild"))

# View severely skewed variables
cat("\n=== SEVERELY SKEWED VARIABLES (|skewness| > 2) ===\n")
severe <- skewness_results %>% 
  filter(severity == "Severe") %>%
  arrange(desc(abs(skewness)))
print(severe)

# View moderately skewed
cat("\n=== MODERATELY SKEWED VARIABLES (|skewness| 1-2) ===\n")
moderate <- skewness_results %>% 
  filter(severity == "Moderate") %>%
  arrange(desc(abs(skewness)))
print(moderate)



## Boxplot findings

03 metals
- Most symmetric and well-separated
- K: Has outliers (Bog and Fen)
- S: Several outliers in Marsh

04 nutrients
- All symmetric, good separation
- NH3_total: Has outliers top and bottom
- Minimal skewness overall

04 gen chem 
- TDS is higher in bogs likely due to water sampling method = justification for dropping TDS
- Alkalinity and hardness separates bog and fen from marsh and swamp
- SO4 means are sep across classes
- None are highly skewed

05 ysi
- pH: bog is skewed with outlier, and overlapping with fen (altough not the means are not)
- pH: Marsh and swamp have overlapping means separa from Bog and Fen
- SpC: Marsh is skewed, swamp has a large range and outlier. Bog and Fen overlap but not the means
- DO[] and DO% show the same distribution and separates them all most bog and Marsh from Swamp and Fen
- ORP: marsh highly skewed means are separate in Fen, Marsh, and Swamp and 3 are very separate from Bog

## Skewness Assessment

- No severely skewed variables (|skewness| > 2)
- 15/41 moderately skewed (|skewness| 1-2):
  - Dissolved O in Swamp/Fen (1.4-2.0) 
  - SpC in Marsh (1.4) - reflects water source heterogeneity
  - Trace metals (Co, Rb, Pb, Mn, Ti, Li, U) show slight skew (1.1-1.4) - typical for low-concentration elements
  - pH in Swamp (-1.6) - mild negative skew
- **Decision**: Proceed with RDA using scale = TRUE. Skewness levels are acceptable and reflect real ecological variation rather than data quality issues.



# UAV Vegetation Structure

In [ ]:
# ============================================================================
# Cell 3: UAV Structure (ONE OBSERVATION PER SITE)
# ============================================================================

# Filter: Wetlands only + unique sites (UAV measured once per site)
df_wetland_unique <- df %>% 
  filter(Class != "UL") %>%
  group_by(SiteID) %>%
  slice(1) %>%
  ungroup()

cat("Wetland sites (unique):", nrow(df_wetland_unique), "\n")
cat("Classes:", paste(unique(df_wetland_unique$Class), collapse = ", "), "\n")

# UAV Structure variables
structure_vars <- c("average_veg_height", "median_veg_height", "max_veg_height",
                    "stddev_veg_height", "P95_veg_height", "P75_veg_height", "P25_veg_height",
                    "CanopyCover1m", "CanopyCover2m", "stem_density")

structure_vars <- structure_vars[structure_vars %in% names(df_wetland_unique)]

structure_long <- df_wetland_unique %>%
  select(Class, all_of(structure_vars)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

# Boxplot
ggplot(structure_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) + 
  facet_wrap(~ variable, scales = "free_y", ncol = 3) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "bottom") +
  labs(title = "UAV Structure Variables by Wetland Class (Unique Sites)",
       y = "Value")

ggsave("Figures/06_structure_distributions.png", width = 12, height = 10)

# Skewness calculation (on unique sites)
skewness_structure <- data.frame()

for(var in structure_vars) {
  if(var %in% names(df_wetland_unique)) {
    for(class in unique(df_wetland_unique$Class)) {
      values <- df_wetland_unique %>% filter(Class == class) %>% pull(var)
      values <- values[!is.na(values)]
      
      if(length(values) > 2) {
        skew_val <- skewness(values)
        
        skewness_structure <- rbind(skewness_structure, 
                                    data.frame(variable = var,
                                              class = class,
                                              skewness = skew_val,
                                              n = length(values)))
      }
    }
  }
}

skewness_structure$severity <- ifelse(abs(skewness_structure$skewness) > 2, "Severe",
                                       ifelse(abs(skewness_structure$skewness) > 1, "Moderate", "Mild"))

cat("\n=== STRUCTURE: SEVERELY SKEWED (|skewness| > 2) ===\n")
print(skewness_structure %>% filter(severity == "Severe") %>% arrange(desc(abs(skewness))))

cat("\n=== STRUCTURE: MODERATELY SKEWED (|skewness| 1-2) ===\n")
print(skewness_structure %>% filter(severity == "Moderate") %>% arrange(desc(abs(skewness))))

# write.csv(skewness_structure, "Figures/08_skewness_structure_CORRECTED.csv", row.names = FALSE)
cat("\n✓ Corrected for multiple observations per site\n")

## Veg struct findings

Most classes have low vegetation structure. But bogs have slight higher veg heights and stem density values. Bogs also have a much greater vegetation height variation than the other three. 

Average_veg_height has one major outlier in the swamp class - which ELA

Conclusion:
-----------
- Most wetlands show very low vegetation structure (median height <0.2m)
- Fen, Marsh, Swamp are indistinguishable by structure (all herbaceous/open)
- Structure variables alone insufficient for wetland classification
  - Need spectral variables (NDVI, NIR) to distinguish herbaceous wetland types

# UAV Multispectral

In [ ]:
# ============================================================================
# Cell 4: UAV Multispectral Analysis (Unique Sites)
# ============================================================================

# Use df_wetland_unique from Cell 3

# ---- PLOT 1: Vegetation Indices ----
indices <- c("NDVI_MEAN", "NDVI_MEDIAN", "NDGVI_MEAN", "NDGVI_MEDIAN",
             "SR_MEAN", "SR_MEDIAN")
indices <- indices[indices %in% names(df_wetland_unique)]

indices_long <- df_wetland_unique %>%
  select(Class, all_of(indices)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

ggplot(indices_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) +
  facet_wrap(~ variable, scales = "free_y", ncol = 3) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Vegetation Indices by Wetland Class")

ggsave("Figures/07a_indices.png", width = 10, height = 6)

# ---- PLOT 2: Raw Spectral Bands ----
bands_central <- c("RED_mean", "RED_median",
                   "GREEN_mean", "GREEN_median", 
                   "REDEDGE_mean", "REDEDGE_median",
                   "NIR_mean", "NIR_median")
bands_central <- bands_central[bands_central %in% names(df_wetland_unique)]

bands_long <- df_wetland_unique %>%
  select(Class, all_of(bands_central)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

ggplot(bands_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) +
  facet_wrap(~ variable, scales = "free_y", ncol = 4) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Raw Spectral Bands by Wetland Class")

ggsave("Figures/07b_bands.png", width = 12, height = 6)

# ---- PLOT 3: Variability Metrics ----
variability <- c("NDVI_STD", "SR_STD", "RED_stdev", "GREEN_stdev", 
                 "REDEDGE_stdev", "NIR_stdev")
variability <- variability[variability %in% names(df_wetland_unique)]

var_long <- df_wetland_unique %>%
  select(Class, all_of(variability)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

ggplot(var_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) +
  facet_wrap(~ variable, scales = "free_y", ncol = 3) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Spectral Variability by Wetland Class")

ggsave("Figures/07c_variability.png", width = 10, height = 6)

# ---- Skewness for ALL spectral variables ----
all_spectral <- c(indices, bands_central, variability)

skewness_spectral <- data.frame()

for(var in all_spectral) {
  if(var %in% names(df_wetland_unique)) {
    for(class in unique(df_wetland_unique$Class)) {
      values <- df_wetland_unique %>% filter(Class == class) %>% pull(var)
      values <- values[!is.na(values)]
      
      if(length(values) > 2) {
        skew_val <- skewness(values)
        skewness_spectral <- rbind(skewness_spectral, 
                                   data.frame(variable = var,
                                             class = class,
                                             skewness = skew_val,
                                             n = length(values)))
      }
    }
  }
}

skewness_spectral$severity <- ifelse(abs(skewness_spectral$skewness) > 2, "Severe",
                                      ifelse(abs(skewness_spectral$skewness) > 1, "Moderate", "Mild"))

cat("\n=== SPECTRAL: SEVERELY SKEWED (|skewness| > 2) ===\n")
print(skewness_spectral %>% filter(severity == "Severe") %>% arrange(desc(abs(skewness))))

cat("\n=== SPECTRAL: MODERATELY SKEWED (|skewness| 1-2) ===\n")
print(skewness_spectral %>% filter(severity == "Moderate") %>% arrange(desc(abs(skewness))))

#write.csv(skewness_spectral, "Figures/10_skewness_spectral.csv", row.names = FALSE)
cat("\n✓ Saved 3 spectral distribution plots and skewness\n")

## UAV Spectral Analysis Findings

### Vegetation Indices
- NDVI and NDGVI show strong class separation (Marsh > Swamp ≈ Fen > Bog)
- Mean and median nearly identical → symmetric distributions
- **SR shows extreme variability and outliers** → exclude from models

### Raw Spectral Bands
- NIR highest in Swamp/Fen (dense vegetation), lowest in Marsh (water influence)
- RED band inversely related to vegetation density (Bog < Fen < Marsh)
- Individual bands show class differences but less separation than NDVI

### Spectral Heterogeneity
- Marsh sites most variable (patchy vegetation/water mosaic)
- Bog/Fen more uniform (consistent vegetation cover)
- Heterogeneity metrics (NDVI_STD, NIR_stdev) are additional predictors

### Conclusion
- **Spectral variables show much better class separation than structure variables**
- NDVI/NDGVI patterns mirror hydrogeochemical gradients
- Evidence that UAV spectral measures DO capture underlying wetland chemistry differences
- SR is extremely variable when open is present (Marshes and slight less but swamps also) 
- NDVI and GNDVI better accout for this (low NIR in water and High NIR reflect in veg)

# Topo Vars

In [ ]:
# ============================================================================
# Cell 5: Topographic Variables (All Classes Including Upland)
# ============================================================================

# Use unique sites for ALL classes (including upland)
df_unique <- df %>%
  group_by(SiteID) %>%
  slice(1) %>%
  ungroup()

cat("Total unique sites:", nrow(df_unique), "\n")
cat("Classes:", paste(unique(df_unique$Class), collapse = ", "), "\n")

# Topographic variables (NO DDS - only DDG!)
topo_vars <- c("log_DDG_mean", "log_DDG_median",
               "log_MCA_median", "log_MCA_mean", "log_MCA_min", "log_MCA_max", 
               "log_MCA_stdev", "log_MCA_p90",
               "log_VDCN_median", "log_VDCN_mean", "log_VDCN_min", "log_VDCN_max",
               "log_VDCN_stdev", "log_VDCN_p90",
               "HOFD_median", "HOFD_mean", "HOFD_min", "HOFD_max", "HOFD_stdev", "HOFD_p90",
               "VOFD_median", "VOFD_mean", "VOFD_min", "VOFD_max", "VOFD_stdev", "VOFD_p90",
               "SWI_median", "SWI_mean", "SWI_min", "SWI_max", "SWI_stdev", "SWI_p90")

topo_vars <- topo_vars[topo_vars %in% names(df_unique)]

topo_long <- df_unique %>%
  select(Class, all_of(topo_vars)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

# Boxplot
ggplot(topo_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) +
  facet_wrap(~ variable, scales = "free_y", ncol = 5) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(size = 7)) +
  labs(title = "Topographic Variables by Class (All Sites)",
       subtitle = "DDG = dist to drainage, MCA = catchment area, VDCN = valley depth, HOFD/VOFD = flow distance")

ggsave("Figures/08_topo_distributions.png", width = 16, height = 10)

# Skewness
skewness_topo <- data.frame()

for(var in topo_vars) {
  if(var %in% names(df_unique)) {
    for(class in unique(df_unique$Class)) {
      values <- df_unique %>% filter(Class == class) %>% pull(var)
      values <- values[!is.na(values)]
      
      if(length(values) > 2) {
        skew_val <- skewness(values)
        skewness_topo <- rbind(skewness_topo, 
                               data.frame(variable = var,
                                         class = class,
                                         skewness = skew_val,
                                         n = length(values)))
      }
    }
  }
}

skewness_topo$severity <- ifelse(abs(skewness_topo$skewness) > 2, "Severe",
                                  ifelse(abs(skewness_topo$skewness) > 1, "Moderate", "Mild"))

cat("\n=== TOPO: SEVERELY SKEWED (|skewness| > 2) ===\n")
print(skewness_topo %>% filter(severity == "Severe") %>% arrange(desc(abs(skewness))))

cat("\n=== TOPO: MODERATELY SKEWED (|skewness| 1-2) ===\n")
print(skewness_topo %>% filter(severity == "Moderate") %>% arrange(desc(abs(skewness))))

#write.csv(skewness_topo, "Figures/11_skewness_topo.csv", row.names = FALSE)
cat("\n✓ Saved topo distributions and skewness\n")

## Topo Findings

- **Strong class separation**: SWI (wetness gradient), HOFD (bog isolation), DDG (drainage proximity)
- **Only 1 severely skewed variable**: HOFD_stdev in Upland
- **Topographic position matches hydrochemical gradients**
- **Most useful predictors**: SWI, HOFD, log_DDG

# Sat remotes

In [ ]:
# ============================================================================
# Cell 6: Satellite Remote Sensing Indices (All Classes Including Upland)
# ============================================================================

# Use same df_unique from Cell 5

# Satellite remote sensing variables
sat_vars <- c("NDVI_amp_harmonic", "NDVI_phase_harmonic", "NDVI_mean_harmonic",
              "TCW_p10", "TCW_p40", "TCW_p80", "TCW_p90")

sat_vars <- sat_vars[sat_vars %in% names(df_unique)]

sat_long <- df_unique %>%
  select(Class, all_of(sat_vars)) %>%
  pivot_longer(cols = -Class, names_to = "variable", values_to = "value")

# Boxplot
ggplot(sat_long, aes(x = Class, y = value, fill = Class)) +
  geom_boxplot() +
  scale_fill_manual(values = class_colors) +
  facet_wrap(~ variable, scales = "free_y", ncol = 3) +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Satellite Remote Sensing Indices by Class (All Sites)",
       subtitle = "NDVI harmonics (seasonal patterns), Tasseled Cap Wetness (TCW)")

ggsave("Figures/09_sat_distributions.png", width = 10, height = 8)

# Skewness
skewness_sat <- data.frame()

for(var in sat_vars) {
  if(var %in% names(df_unique)) {
    for(class in unique(df_unique$Class)) {
      values <- df_unique %>% filter(Class == class) %>% pull(var)
      values <- values[!is.na(values)]
      
      if(length(values) > 2) {
        skew_val <- skewness(values)
        skewness_sat <- rbind(skewness_sat, 
                              data.frame(variable = var,
                                        class = class,
                                        skewness = skew_val,
                                        n = length(values)))
      }
    }
  }
}

skewness_sat$severity <- ifelse(abs(skewness_sat$skewness) > 2, "Severe",
                                 ifelse(abs(skewness_sat$skewness) > 1, "Moderate", "Mild"))

cat("\n=== SAT REMOTE SENSING: SEVERELY SKEWED (|skewness| > 2) ===\n")
print(skewness_sat %>% filter(severity == "Severe") %>% arrange(desc(abs(skewness))))

cat("\n=== SAT REMOTE SENSING: MODERATELY SKEWED (|skewness| 1-2) ===\n")
print(skewness_sat %>% filter(severity == "Moderate") %>% arrange(desc(abs(skewness))))

#write.csv(skewness_sat, "Figures/12_skewness_sat.csv", row.names = FALSE)
cat("\n✓ Saved satellite remote sensing distributions and skewness\n")

# SPLOMs - within group

In [ ]:
# ============================================================================
# Cell 7: Within-Group Correlations (Professional SPLOMs)
# ============================================================================

# Custom panel functions for better SPLOMs
panel.cor <- function(x, y, digits = 3, prefix = "", cex.cor, ...) {
  usr <- par("usr"); on.exit(par(usr))
  par(usr = c(0, 1, 0, 1))
  r <- cor(x, y, use = "complete.obs")
  txt <- format(c(r, 0.123456789), digits = digits)[1]
  txt <- paste0(prefix, txt)
  text(0.5, 0.5, txt, cex = 2)
}

panel.hist <- function(x, ...) {
  usr <- par("usr"); on.exit(par(usr))
  par(usr = c(usr[1:2], 0, 1.5))
  h <- hist(x, plot = FALSE)
  breaks <- h$breaks; nB <- length(breaks)
  y <- h$counts; y <- y/max(y)
  rect(breaks[-nB], 0, breaks[-1], y, col = "lightblue")
}

# ---- 7.1: MULTISPECTRAL BANDS ----
cat("\n=== MULTISPECTRAL BAND CORRELATIONS ===\n")

multispectral_check <- c("RED_mean", "RED_median", 
                         "GREEN_mean", "GREEN_median",
                         "REDEDGE_mean", "REDEDGE_median",
                         "NIR_mean", "NIR_median")
multispectral_check <- multispectral_check[multispectral_check %in% names(df_wetland_unique)]

cor_multispectral <- cor(df_wetland_unique[, multispectral_check], use = "pairwise.complete.obs")
print(round(cor_multispectral, 3))

png("Figures/10a_multispectral_correlations.png", width = 1400, height = 1400, res = 150)
pairs(df_wetland_unique[, multispectral_check],
      main = "Multispectral Band Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19)
dev.off()
cat("✓ Saved: Figures/10a_multispectral_correlations.png\n")

# ---- 7.2: STRUCTURE VARIABLES ----
cat("\n=== STRUCTURE VARIABLE CORRELATIONS ===\n")

structure_check <- c("average_veg_height", "median_veg_height", 
                     "P95_veg_height", "P75_veg_height",
                     "CanopyCover1m", "CanopyCover2m")
structure_check <- structure_check[structure_check %in% names(df_wetland_unique)]

cor_structure <- cor(df_wetland_unique[, structure_check], use = "pairwise.complete.obs")
print(round(cor_structure, 3))

png("Figures/10b_structure_correlations.png", width = 1200, height = 1200, res = 150)
pairs(df_wetland_unique[, structure_check],
      main = "Structure Variable Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19)
dev.off()
cat("✓ Saved: Figures/10b_structure_correlations.png\n")

# ---- 7.3: VEGETATION INDICES ----
cat("\n=== VEGETATION INDEX CORRELATIONS ===\n")

indices_check <- c("NDVI_MEAN", "NDVI_MEDIAN", 
                   "NDGVI_MEAN", "NDGVI_MEDIAN",
                   "SR_MEAN", "SR_MEDIAN")
indices_check <- indices_check[indices_check %in% names(df_wetland_unique)]

cor_indices <- cor(df_wetland_unique[, indices_check], use = "pairwise.complete.obs")
print(round(cor_indices, 3))

png("Figures/10c_indices_correlations.png", width = 1200, height = 1200, res = 150)
pairs(df_wetland_unique[, indices_check],
      main = "Vegetation Index Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19)
dev.off()
cat("✓ Saved: Figures/10c_indices_correlations.png\n")

# ---- 7.4: TOPOGRAPHIC FLOW DISTANCES ----
cat("\n=== FLOW DISTANCE CORRELATIONS ===\n")

flow_check <- c("HOFD_mean", "HOFD_median", "HOFD_p90",
                "VOFD_mean", "VOFD_median", "VOFD_p90")
flow_check <- flow_check[flow_check %in% names(df_unique)]

cor_flow <- cor(df_unique[, flow_check], use = "pairwise.complete.obs")
print(round(cor_flow, 3))

png("Figures/10d_flow_distance_correlations.png", width = 1200, height = 1200, res = 150)
pairs(df_unique[, flow_check],
      main = "Flow Distance Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19)
dev.off()
cat("✓ Saved: Figures/10d_flow_distance_correlations.png\n")

# ---- 7.5: MAJOR CATIONS ----
cat("\n=== CATION CORRELATIONS ===\n")

cations_check <- c("Ca", "Mg", "Na", "K")
cations_check <- cations_check[cations_check %in% names(df)]

cor_cations <- cor(df[, cations_check], use = "pairwise.complete.obs")
print(round(cor_cations, 3))

png("Figures/10e_cation_correlations.png", width = 1000, height = 1000, res = 150)
pairs(df[, cations_check],
      main = "Major Cation Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19)
dev.off()
cat("✓ Saved: Figures/10e_cation_correlations.png\n")

cat("\n=== CORRELATION ANALYSIS COMPLETE ===\n")

# ============================================================================
# Cell 7.6: Chemistry Group Correlations
# ============================================================================

# ---- 7.6a: Dissolved Metals (Major Elements) ----
cat("\n=== MAJOR DISSOLVED METAL CORRELATIONS ===\n")

# Focus on common/abundant metals
major_metals <- c("Ca", "Mg", "Na", "K", "Fe", "Mn", "Al")
major_metals <- major_metals[major_metals %in% names(df)]

cor_major_metals <- cor(df[, major_metals], use = "pairwise.complete.obs")
print(round(cor_major_metals, 3))

png("Figures/10f_major_metals_correlations.png", width = 1400, height = 1400, res = 150)
pairs(df[, major_metals],
      main = "Major Dissolved Metals Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.6)
dev.off()
cat("✓ Saved: Figures/10f_major_metals_correlations.png\n")

# ---- 7.6b: Trace Metals ----
cat("\n=== TRACE METAL CORRELATIONS ===\n")

trace_metals <- c("Cu", "Zn", "Pb", "Ni", "Co", "Li")
trace_metals <- trace_metals[trace_metals %in% names(df)]

cor_trace_metals <- cor(df[, trace_metals], use = "pairwise.complete.obs")
print(round(cor_trace_metals, 3))

png("Figures/10g_trace_metals_correlations.png", width = 1200, height = 1200, res = 150)
pairs(df[, trace_metals],
      main = "Trace Metals Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.6)
dev.off()
cat("✓ Saved: Figures/10g_trace_metals_correlations.png\n")

# ---- 7.6c: Nutrients ----
cat("\n=== NUTRIENT CORRELATIONS ===\n")

nutrients <- c("DOC", "TOC", "P_dissolved", "P_total", "NH3_total")
nutrients <- nutrients[nutrients %in% names(df)]

cor_nutrients <- cor(df[, nutrients], use = "pairwise.complete.obs")
print(round(cor_nutrients, 3))

png("Figures/10h_nutrients_correlations.png", width = 1000, height = 1000, res = 150)
pairs(df[, nutrients],
      main = "Nutrient Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.7)
dev.off()
cat("✓ Saved: Figures/10h_nutrients_correlations.png\n")

# ---- 7.6d: General Chemistry & YSI ----
cat("\n=== GENERAL CHEMISTRY & YSI CORRELATIONS ===\n")

general_ysi <- c("pH", "SpC", "Alkalinity_total_CaCO3_mg.L", "Hardness", 
                 "TDS_mg.L", "Dissolved_O_mgL", "ORPmV")
general_ysi <- general_ysi[general_ysi %in% names(df)]

cor_general <- cor(df[, general_ysi], use = "pairwise.complete.obs")
print(round(cor_general, 3))

png("Figures/10i_general_chemistry_correlations.png", width = 1400, height = 1400, res = 150)
pairs(df[, general_ysi],
      main = "General Chemistry & YSI Correlations",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.6)
dev.off()
cat("✓ Saved: Figures/10i_general_chemistry_correlations.png\n")

cat("\n=== CHEMISTRY CORRELATION ANALYSIS COMPLETE ===\n")

## Findings from SPLOMs

### MS bands - very Redundant
- Mean vs median = identical (r = 0.944 to 0.997)
- Green adn Red are highly correelated
- Rededge adn NIR and highly correlated
** Drop medians and keep means

### Structure vars - very REDUNDANCT! 
** Keep only average_veg_height (or median), CanopyCover1m, stem_density ?
** Drop P95_veg_height, P75_veg_height, CanopyCover2m

### Veg indices 
- Mean and medians are the same drop one
- NDVI vs GNDVI = mod correlated
- SR vs NDVI/ GNDVI = weakly correlated SR is measuring something different than NDVI and GNDVI 
  - BUT SR as the issue with variability
**  KEEP: NDVI_MEAN, NDGVI_MEAN ~ Unsure what to do about SR 

### Flow distances (Remember that Upland was included here - Re-run when doing RDA2 with Topo and Sat remotes)
** KEEP: HOFD_mean (or median), VOFD_mean (or median)
** DROP: HOFD_p90, HOFD_median (redundant with mean), VOFD_p90

### Major Cations 
- Ca and Mg = very correlated but could be kept because of domain knowledge and helping with interp
- Ca/Mg vs Na
- K low correlations with the other
** Keep all

### Nutrients
- P diss and NH3 = highly corr
- DOC and TOC = highly corr
** DROP TDS


# SPLOMs - Between groups 

In [ ]:
# ============================================================================
# Cell 8: Cross-Group Correlations (Research Question)
# ============================================================================
# Research Question: Can UAV measures capture hydrogeochemical characteristics?

# Use df_wetland_unique for UAV variables (wetlands only, unique sites)

# ---- 8.1: UAV Spectral vs. Key Chemistry ----
cat("\n=== UAV SPECTRAL vs. HYDROCHEMISTRY ===\n")

# Select key variables
uav_spectral <- c("NDVI_MEAN", "NDGVI_MEAN", "NIR_mean", "RED_mean")
key_chemistry <- c("pH", "Ca", "Mg", "DOC", "SpC", "Alkalinity_total_CaCO3_mg.L")

uav_spectral <- uav_spectral[uav_spectral %in% names(df_wetland_unique)]
key_chemistry <- key_chemistry[key_chemistry %in% names(df_wetland_unique)]

# Combine for correlation
uav_chem <- c(uav_spectral, key_chemistry)

cor_uav_chem <- cor(df_wetland_unique[, uav_chem], use = "pairwise.complete.obs")
print(round(cor_uav_chem, 3))

png("Figures/11a_uav_spectral_vs_chemistry.png", width = 1400, height = 1400, res = 150)
pairs(df_wetland_unique[, uav_chem],
      main = "UAV Spectral vs. Hydrochemistry\n(Wetlands Only)",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.7)
dev.off()
cat("✓ Saved: Figures/11a_uav_spectral_vs_chemistry.png\n")

# ---- 8.2: UAV Structure vs. Chemistry ----
cat("\n=== UAV STRUCTURE vs. HYDROCHEMISTRY ===\n")

uav_structure <- c("average_veg_height", "CanopyCover1m", "stem_density")
uav_structure <- uav_structure[uav_structure %in% names(df_wetland_unique)]

structure_chem <- c(uav_structure, key_chemistry)

cor_structure_chem <- cor(df_wetland_unique[, structure_chem], use = "pairwise.complete.obs")
print(round(cor_structure_chem, 3))

png("Figures/11b_uav_structure_vs_chemistry.png", width = 1200, height = 1200, res = 150)
pairs(df_wetland_unique[, structure_chem],
      main = "UAV Structure vs. Hydrochemistry\n(Wetlands Only)",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.7)
dev.off()
cat("✓ Saved: Figures/11b_uav_structure_vs_chemistry.png\n")

# ---- 8.3: Topography (SWI, HOFD) vs. Chemistry ----
cat("\n=== TOPOGRAPHY vs. HYDROCHEMISTRY ===\n")

topo_key <- c("SWI_mean", "HOFD_mean", "log_DDG_mean")
topo_key <- topo_key[topo_key %in% names(df_unique)]

# Use df_unique (all sites) for topography
topo_chem_vars <- c(topo_key, key_chemistry)
topo_chem_vars <- topo_chem_vars[topo_chem_vars %in% names(df_unique)]

cor_topo_chem <- cor(df_unique[, topo_chem_vars], use = "pairwise.complete.obs")
print(round(cor_topo_chem, 3))

png("Figures/11c_topography_vs_chemistry.png", width = 1200, height = 1200, res = 150)
pairs(df_unique[, topo_chem_vars],
      main = "Topography vs. Hydrochemistry\n(All Sites)",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.7)
dev.off()
cat("✓ Saved: Figures/11c_topography_vs_chemistry.png\n")

# ---- 8.4: UAV Spectral vs. Field Vegetation ----
cat("\n=== UAV SPECTRAL vs. FIELD VEGETATION ===\n")

field_veg <- c("tall_tree", "small_tree", "shrub", "graminoid", "moss")
field_veg <- field_veg[field_veg %in% names(df_wetland_unique)]

uav_field <- c(uav_spectral, field_veg)
uav_field <- uav_field[uav_field %in% names(df_wetland_unique)]

cor_uav_field <- cor(df_wetland_unique[, uav_field], use = "pairwise.complete.obs")
print(round(cor_uav_field, 3))

png("Figures/11d_uav_spectral_vs_field_veg.png", width = 1400, height = 1400, res = 150)
pairs(df_wetland_unique[, uav_field],
      main = "UAV Spectral vs. Field Vegetation\n(Wetlands Only)",
      lower.panel = points,
      diag.panel = panel.hist,
      upper.panel = panel.cor,
      pch = 19,
      cex = 0.7)
dev.off()
cat("✓ Saved: Figures/11d_uav_spectral_vs_field_veg.png\n")

cat("\n=== CROSS-GROUP CORRELATION ANALYSIS COMPLETE ===\n")
cat("Review plots to assess research question:\n")
cat("Do UAV spectral/structure measures correlate with hydrochemistry?\n\n")

## Findings

# UAV MS data vs chem 
- Strong neg correlations 
  - NDVi_mean vs NIR_mean = r -.8 highNIR = lower NDVI ()

In [ ]:
# Quick diagnostic to check if data makes sense
cat("\n=== NDVI-NIR RELATIONSHIP DIAGNOSTIC ===\n")

# Select just NDVI and band data
diag_data <- df_wetland_unique %>%
  select(SiteID, Class, NDVI_MEAN, NDVI_MEDIAN, NIR_mean, NIR_median, RED_mean, RED_median) %>%
  na.omit()

# Calculate NDVI manually from NIR and RED means
diag_data$NDVI_calculated <- (diag_data$NIR_mean - diag_data$RED_mean) / 
                              (diag_data$NIR_mean + diag_data$RED_mean)

# Compare stored NDVI with calculated NDVI
cat("\nCorrelation between stored NDVI_MEAN and manually calculated NDVI:\n")
cor_check <- cor(diag_data$NDVI_MEAN, diag_data$NDVI_calculated)
print(paste("r =", round(cor_check, 3)))

# If they don't match (r < 0.9), there's a data issue
if(cor_check < 0.9) {
  cat("\n⚠️ WARNING: Stored NDVI does not match calculated NDVI!\n")
  cat("This suggests NDVI_MEAN is calculated from pixel-level NDVI (correct),\n")
  cat("while NIR_mean/RED_mean are band averages (also correct).\n")
  cat("This is EXPECTED and explains the negative correlation.\n\n")
}

# Check by class
cat("\nMean values by class:\n")
summary_by_class <- diag_data %>%
  group_by(Class) %>%
  summarise(
    NDVI = mean(NDVI_MEAN),
    NIR = mean(NIR_mean),
    RED = mean(RED_mean),
    n = n()
  )
print(summary_by_class)

# Look at raw scatterplot
png("Figures/DIAGNOSTIC_ndvi_nir.png", width = 800, height = 800, res = 120)
plot(diag_data$NIR_mean, diag_data$NDVI_MEAN, 
     col = as.factor(diag_data$Class),
     pch = 19,
     xlab = "NIR_mean (band average)",
     ylab = "NDVI_MEAN (pixel-level NDVI average)",
     main = "NDVI vs NIR Relationship by Class")
legend("topright", legend = levels(as.factor(diag_data$Class)), 
       col = 1:length(levels(as.factor(diag_data$Class))), pch = 19)
abline(lm(NDVI_MEAN ~ NIR_mean, data = diag_data), col = "red", lwd = 2)
dev.off()

cat("\n✓ Saved diagnostic plot: Figures/DIAGNOSTIC_ndvi_nir.png\n")